## File: `timeit_Algo-1.ipynb`  
**Project:** HHG-SaDAS  

### Code Description  
This notebook measures the CPU time required to execute the code.  

Following Appendix-A of my thesis:  
- **Algo-1** uses a large, equispaced grid in the interval $x \in (-1, 1)$  
  and directly estimates the roots of $P'_N(x)$ from the local maxima  
  of $-P'_N(x)^2$ (serving as the initial guesses).  
- It does not save data; rather, it shows detailed execution time analysis.

---

**Author:** Siddhartha Mithiya  
**Affiliation:** Indian Institute of Technology (IIT) Mandi  
**License:** MIT License  
**Repository:** [https://github.com/Siddhartha-Acad/HHG-SaDAS.git](https://github.com/Siddhartha-Acad/HHG-SaDAS.git)  

---

### Notes  
- Collocation points are calculated using the zeros of $P'_N$ (analytical derivative) with the `fsolve` function of SciPy.  
- This notebook is part of the HHG-SaDAS package, developed during my MS(R) thesis:  
  *"Higher-Order Harmonic Generation and Harmonic-Power Enhancement in Noble-Gas Atoms Confined Inside C60".*


In [1]:
import time
import warnings
import numpy as np
import pandas as pd
from scipy.misc import derivative
from scipy.optimize import fsolve
from scipy.special import legendre
from scipy.signal import find_peaks

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
def P_N(x):
    return legendre(N)(x)

def P_N_deriv(x):
    return derivative(P_N, float(x), dx * 0.01, 1)

In [3]:
N = 6
nop_per_segment = 210
nopx = nop_per_segment * 10
total_segments = nopx // nop_per_segment
x_segment_pts = np.linspace(-1, 1, total_segments + 1)
dx = float((x_segment_pts[1] - x_segment_pts[0]) / (nop_per_segment - 1))

if N % 2 == 0:
    all_roots = [0]
else:
    all_roots = []

In [4]:
for i in range(total_segments):
    segment_start_time = time.time()
    print(f'Segment : {i}')
    x_segment = np.linspace(x_segment_pts[i], x_segment_pts[i + 1], nop_per_segment)

    print('calculating normal derivative for guess points.')
    PN_deriv_array = np.array([derivative(P_N, float(xi), dx, 1) for xi in x_segment])

    pks_at = find_peaks(-PN_deriv_array ** 2)[0]
    print('calculating collocation points in this segment.\n')
    colloc_pt_segment = np.array(
        [fsolve(P_N_deriv, x_segment[pks_at[j]], xtol=10 ** -15)[0] for j in range(len(pks_at))])

    all_roots.extend(colloc_pt_segment)

all_roots = sorted(all_roots)

print(all_roots)
print('Number of collocation points: ', len(all_roots))


Segment : 0
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 1
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 2
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 3
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 4
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 5
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 6
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 7
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 8
calculating normal derivative for guess points.
calculating collocation points in this segment.

Segment : 9
calcula

In [5]:
%%timeit
for i in range(total_segments):
    segment_start_time = time.time()
    # print(f'Segment : {i}')
    x_segment = np.linspace(x_segment_pts[i], x_segment_pts[i + 1], nop_per_segment)

    # print('calculating normal derivative for guess points.')
    PN_deriv_array = np.array([derivative(P_N, float(xi), dx, 1) for xi in x_segment])

    pks_at = find_peaks(-PN_deriv_array ** 2)[0]
    # print('calculating collocation points in this segment.\n')
    colloc_pt_segment = np.array(
        [fsolve(P_N_deriv, x_segment[pks_at[j]], xtol=10 ** -15)[0] for j in range(len(pks_at))])

    all_roots.extend(colloc_pt_segment)

1.04 s ± 90.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
